In [0]:
%pip install google-cloud-storage

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../tests/test_silver_layer

In [0]:
%run ../utils/gcp_setup

In [0]:
import os
from google.cloud import storage
import pyspark.sql.functions as F
import logging
import sys

# 1. Setup the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("HolidaysBronzeIngestion")

**Databricks Widgets Set Up**

In [0]:
dbutils.widgets.text("credentials_path", "")
dbutils.widgets.text("gcs_bucket_name", "")
dbutils.widgets.text("gcs_weather_file_path", "")
dbutils.widgets.text("dbfs_chicago_weather_path", "")
dbutils.widgets.text("bronze_weather_table", "")

credentials_path = dbutils.widgets.get("credentials_path") or "/Workspace/Shared/service-account.json"
gcs_bucket_name = dbutils.widgets.get("gcs_bucket_name") or "prefect-bucket-latypov"
gcs_weather_file_path = dbutils.widgets.get("gcs_weather_file_path") or "chicago_weather_data/chicago_weather_2025_2026.parquet"
dbfs_chicago_weather_path = dbutils.widgets.get("dbfs_chicago_weather_path") or f"/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026"
bronze_weather_table = dbutils.widgets.get("bronze_weather_table") or "chicago_taxi_data.bronze.bronze_weather"

**Google Cloud credentials Set Up**

In [0]:
logger.info(f"Set up Google Cloud service account keys")
setup_gcp_creds(credentials_path=credentials_path)

2026-09-09 10:58:11 - INFO - Set up Google Cloud service account keys
2026-09-09 10:58:11 - INFO - Found service account file
2026-09-09 10:58:12 - INFO - Successfully authenticated. Found 2 buckets.


**Chicago Weather Data 2025-2026 years Ingestion to Databricks Unity Catalog Volume**

In [0]:

try:
    # 1. Setup GCS client
    client = storage.Client()


    bucket = client.get_bucket(gcs_bucket_name)
    logger.info(f"GCS bucket: {bucket}")
    blob = bucket.blob(gcs_weather_file_path)
    logger.info(f"Blob in GCS bucket: {blob}")

    # 2. Path to Unity Catalog Volume
    os.makedirs(dbfs_chicago_weather_path, exist_ok=True)
    output_filename = gcs_weather_file_path.split('/')[-1]
    logger.info(f"Output filename: {output_filename}")
    destination_path = f"{dbfs_chicago_weather_path}/{output_filename}"
    logger.info(f"Destination Path: {destination_path}")

    # 3. Write to Unity Catalog Volume
    with open(destination_path, "wb") as f:
        blob.download_to_file(f)

    logger.info(f"File with Chicago holiday data ingestested to {destination_path}")
except Exception as e:
    logger.error(f"File ingestestion failed: {e}")
    dbutils.notebook.exit(f"File ingestestion failed: {e}")

**Data Quality Check**

Making automated bronze layer quality checks on the raw incoming GCS data

In [0]:
df_raw_weather = spark.read.parquet(destination_path)

required_columns = ["date", "temp", "precip", "wind_speed", "humidity", "hour"]
try: 
    logger.info("Starting Testing Chicago Weather Data Bronze Ingestion")

    validate_schema(df_raw_weather, required_columns)

    validate_date_range(df_raw_weather, "date", min_date="2025-01-01", max_date="2026-05-01")
    logger.info("DQ passed")
except Exception as e:
    error_message = str(e)
    logger.error(f"Aborting Job: {error_message}")
    
    dbutils.notebook.exit(f"{error_message}\nDQ checks FAILED")



[UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually. SQLSTATE: 42KD9

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.dataSchemaNotSpecifiedError(QueryCompilationErrors.scala:3499)
	at org.apache.spark.sql.execution.datasources.DataSource.$anonfun$getOrInferFileFormatSchema$15(DataSource.scala:303)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.execution.datasources.DataSource.$anonfun$getOrInferFileFormatSchema$1(DataSource.scala:303)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at org.apache.spark.sql.execution.datasources.DataSource.record(DataSource.scala:325)
	at org.apache.spark.sql.execution.datasources.DataSource.getOrInferFile

**Save As Delta Table**

Add **ingestion_timestamp** and **ingestion_timestamp** columns

In [0]:
from delta.tables import DeltaTable

df_bronze_weather = df_raw_weather.withColumn("ingestion_timestamp", F.current_timestamp())\
                                    .withColumn("source_file", F.col("_metadata.file_path"))


# (df_bronze_weather.write.mode("append")             
#     .option("mergeSchema", "true") 
#     .saveAsTable(bronze_weather_table)
# )

# In your Weather Bronze notebook
if spark.catalog.tableExists(bronze_weather_table):
    target_table = DeltaTable.forName(spark, bronze_weather_table)
    (target_table.alias("t")
     .merge(df_bronze_weather.alias("s"), "t.date = s.date AND t.hour = s.hour")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    logger.info("Merge succeessfull")
else:
    df_bronze_weather.write.format("delta").saveAsTable(bronze_weather_table)

In [0]:
display(spark.sql(f"SELECT * FROM {bronze_weather_table} ORDER BY date DESC LIMIT 10"))

date,temp,precip,wind_speed,humidity,year,month,day,hour,ingestion_timestamp,source_file
2026-07-01T23:00:00.000Z,29.7,0.0,16.4,69,2026,7,1,23,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T22:00:00.000Z,30.3,0.0,17.1,68,2026,7,1,22,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T21:00:00.000Z,31.2,0.0,19.5,62,2026,7,1,21,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T20:00:00.000Z,32.0,0.0,18.8,57,2026,7,1,20,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T19:00:00.000Z,33.4,0.0,18.6,51,2026,7,1,19,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T18:00:00.000Z,33.8,0.0,20.1,46,2026,7,1,18,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T17:00:00.000Z,34.9,0.0,18.4,43,2026,7,1,17,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T16:00:00.000Z,35.4,0.0,19.0,39,2026,7,1,16,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T15:00:00.000Z,35.8,0.0,17.8,39,2026,7,1,15,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
2026-07-01T14:00:00.000Z,35.7,0.0,16.9,39,2026,7,1,14,2026-09-09T11:08:20.308Z,dbfs:/Volumes/chicago_taxi_data/bronze/gcs_data/chicago_weather_2025_2026/part-00007-tid-3014008553433999426-45c8753f-dea1-42b9-b6fc-35aba0d21f68-151-1.c000.snappy.parquet
